In [ ]:
%pip install -q transformers datasets

In [ ]:
import os
import torch
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoImageProcessor, AutoModelForImageClassification

In [ ]:
torch.set_grad_enabled(False)

is_kaggle = os.path.exists("/kaggle")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_name = "microsoft/resnet-18"
dataset_name = "huggingface/cats-image"

print(f"Kaggle environment: {is_kaggle}")
print(f"Using device: {device}")
print(f"Model name: {model_name}")
print(f"Dataset name: {dataset_name}")

In [ ]:
try:
    dataset = load_dataset(dataset_name)
    sample = dataset["test"][0]
    image = sample["image"]
    print("Dataset loaded successfully.")
    print(sample)
except Exception as e:
    raise RuntimeError(
        "샘플 이미지 로드에 실패했습니다. Kaggle Notebook이라면 Internet 옵션이 켜져 있는지 확인하세요.\n"
        f"원본 에러: {e}"
    )

In [ ]:
# 불러온 샘플 이미지를 확인합니다.
plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Sample image from datasets")
plt.show()

In [ ]:
# 이미지 전처리기와 분류 모델을 불러옵니다.
try:
    processor = AutoImageProcessor.from_pretrained(model_name)
    model = AutoModelForImageClassification.from_pretrained(model_name).to(device)
    model.eval()
    print("Model and processor loaded successfully.")
except Exception as e:
    raise RuntimeError(
        "모델 또는 프로세서 로드에 실패했습니다. Kaggle Notebook이라면 Internet 옵션을 확인하세요.\n"
        f"원본 에러: {e}"
    )

In [ ]:
# 전처리기가 어떤 입력 규칙을 사용하는지 간단히 확인합니다.
print(processor)

In [ ]:
# AutoImageProcessor를 사용해 이미지를 텐서로 변환합니다.
inputs = processor(images=image, return_tensors="pt")
inputs = {key: value.to(device) for key, value in inputs.items()}

for key, value in inputs.items():
    print(f"{key}: shape={tuple(value.shape)}, dtype={value.dtype}, device={value.device}")

In [ ]:
# PyTorch 기반으로 추론을 수행합니다.
with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits
    probabilities = torch.softmax(logits, dim=-1)

print("Logits shape:", tuple(logits.shape))
print("Probabilities shape:", tuple(probabilities.shape))

In [ ]:
# Top-1 결과를 계산하고 출력합니다.
top1_prob, top1_idx = torch.max(probabilities, dim=-1)
top1_label = model.config.id2label[top1_idx.item()]

print("Top-1 prediction")
print(f"Class: {top1_label}")
print(f"Probability: {top1_prob.item():.4f}")

In [ ]:
# Top-10 예측 결과를 확률 순으로 출력합니다.
topk = 10
topk_probs, topk_indices = torch.topk(probabilities, k=topk, dim=-1)

print(f"Top-{topk} predictions")
for rank, (prob, idx) in enumerate(zip(topk_probs[0], topk_indices[0]), start=1):
    label = model.config.id2label[idx.item()]
    print(f"{rank:2d}. {label:<30} {prob.item():.4f}")